# Travel Capstone MLOps - Modeling

This notebook covers the Data Exploration, Preprocessing, and Model Training for the Travel Capstone Project. We train three models:
1. **Regression**: Flight Price Prediction
2. **Classification**: Gender Prediction
3. **Recommender**: Hotel Recommendation

All runs are tracked using a local MLflow sqlite database.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
import mlflow
import mlflow.sklearn
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set up local MLflow tracking
mlflow.set_tracking_uri('sqlite:///mlflow.db')

In [ ]:
users_df = pd.read_csv('users.csv')
flights_df = pd.read_csv('flights.csv')
hotels_df = pd.read_csv('hotels.csv')

print(f"Users: {users_df.shape}")
print(f"Flights: {flights_df.shape}")
print(f"Hotels: {hotels_df.shape}")

In [ ]:
mlflow.set_experiment("Flight_Price_Regression")
with mlflow.start_run(run_name="RandomForest_Regressor"):
    features = ['flightType', 'time', 'distance', 'agency', 'from', 'to']
    X = flights_df[features]
    y = flights_df['price']
    
    categorical_features = ['flightType', 'agency', 'from', 'to']
    numeric_features = ['time', 'distance']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    
    model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('regressor', RandomForestRegressor(n_estimators=50, random_state=42))])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds, squared=False)
    print(f"Regression RMSE: {rmse}")
    
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, "flight_price_model")
    joblib.dump(model, "flight_price_model.joblib")
    print("Flight model saved to flight_price_model.joblib")

In [ ]:
mlflow.set_experiment("Gender_Classification")
with mlflow.start_run(run_name="RandomForest_Classifier"):
    features = ['company', 'age']
    X = users_df[features].copy()
    y = users_df['gender']
    
    categorical_features = ['company']
    numeric_features = ['age']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    
    model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('classifier', RandomForestClassifier(n_estimators=50, random_state=42))])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"Classification Accuracy: {acc}")
    print(classification_report(y_test, preds))
    
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(model, "gender_classification_model")
    joblib.dump(model, "gender_classification_model.joblib")
    print("Gender classification model saved to gender_classification_model.joblib")

In [ ]:
mlflow.set_experiment("Hotel_Recommender")
with mlflow.start_run(run_name="Popularity_Recommender"):
    hotel_stats = hotels_df.groupby(['name', 'place']).agg(
        total_bookings=('travelCode', 'count'),
        avg_price=('price', 'mean'),
        avg_days=('days', 'mean')
    ).reset_index()
    
    hotel_stats['norm_bookings'] = hotel_stats['total_bookings'] / hotel_stats['total_bookings'].max()
    hotel_stats['norm_price'] = hotel_stats['avg_price'] / hotel_stats['avg_price'].max()
    hotel_stats['score'] = (hotel_stats['norm_bookings'] * 0.7) - (hotel_stats['norm_price'] * 0.3)
    
    recommendations = hotel_stats.sort_values(by=['place', 'score'], ascending=[True, False])
    
    recommender_dict = {}
    for place in recommendations['place'].unique():
        recommender_dict[place] = recommendations[recommendations['place'] == place].head(5)[['name', 'avg_price', 'score']].to_dict('records')
    
    if 'Florianopolis (SC)' in recommender_dict:
        print("Example recommendations for Florianopolis (SC):")
        print(recommender_dict['Florianopolis (SC)'])
    
    mlflow.log_metric("places_covered", len(recommender_dict))
    joblib.dump(recommender_dict, "hotel_recommender.joblib")
    mlflow.log_artifact("hotel_recommender.joblib")
    print("Hotel recommender saved to hotel_recommender.joblib")